Ce notebook contient un test de few shot prompting sur le modèle GPT-oss. J'ai décidé de ne mettre que deux exemples.

# Installation et configuration

In [ ]:
# Uniquement pour Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Uniquement pour Colab
%cd '/content/drive/MyDrive/FAC/Master/M2/Stage/Code'
# path Maïwenn à changer

/content/drive/MyDrive/FAC/Master/M2/Stage/Code


In [ ]:
# Uniquement pour Colab, sinon télécharger Ollama https://ollama.com/download
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,436 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,786 kB]
Get:13 https://r2u.stat.illinois.edu/

In [ ]:
# Uniquement pour Colab
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
!pip install ollama

In [ ]:
import pandas as pd
import ollama
import json

In [ ]:
!ollama pull gpt-oss

# Préparation du fichier jsonl

In [ ]:
df = pd.read_json("./fr-extract.jsonl.gz", lines=True, chunksize=10)
full_df = pd.DataFrame()

for chunk in df :
  try :
     small_df = chunk[chunk["lang"] == "Français"][["word", "pos_title", "senses", "etymology_texts"]]
  except Exception as e :
    print(e)
    print(chunk)
    continue
  # à partir de notre chunk courant on ne prend que certaines colonnes pour créer un df
  small_df["senses"] = small_df["senses"].apply( lambda x: x[0]["glosses"] if isinstance(x, list) and len(x) > 0 and "glosses" in x[0] else None )
  # certaines entrées n'ont pas de gloss apparemment, donc prendre la première gloss (dans le cas où il y en a plusieurs) ou rien du tout s'il n'y en a pas
  small_df = small_df.rename(columns={"word": "mot", "pos_title": "catégorie", "senses": "définition", "etymology_texts": "étymologie"})
  list_of_dict = small_df.to_dict("records")
  # transformer le df en liste de dictionnaires pour en faire un fichier jsonl (meilleur format pour un input de LLM)

  with open("chunk.jsonl", "w", encoding="utf-8") as f :
       for d in list_of_dict :
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

  json_entries = [] # pour pouvoir donner plusieurs entrées au prompt, une à la fois

  with open("chunk.jsonl", "r", encoding="utf-8") as f :
      for line in f :
          json_entry = json.loads(line)
          print(json_entry)
          json_entries.append(json_entry)

  llm_responses = [] # pouvoir stocker les réponses du LLM (en sachant qu'une sortie = une réponse pour un mot)
  bad_llm_responses = []

  SYSTEM_PROMPT = """Tu es un assistant d'extraction d'information.
  Règles :
  - tu ne dois pas donner d'explications, tu ne dois pas raisonner et tu ne dois pas reformuler la tâche
  - tu ne dois rien déduire
  - tu dois répondre uniquement avec un JSON valide
  - il ne doit y avoir aucun texte avant ou après le JSON
  - les réponses sont en minuscules et sans déterminant
  - le JSON doit être sur une seule ligne"""

  for json_entry in json_entries : # json_entry correspond à une ligne du jsonl, qui correspond à un mot, json_entries est la liste de tous les mots du chunk
    prompt = f"""l'enregistrement en JSON suivant : {json_entry} contient un mot du français, une catégorie, une définition et une étymologie. Construis un nouvel enregistrement JSON qui contient les réponses aux questions suivantes :
        * Q1 = quelle est la langue d'origine du mot qui est indiquée dans l'étymologie ? (un seul mot ou NULL si l'information est absente)
        * Q2 = quelle est la base du mot qui est indiquée dans l'étymologie ? (un seul mot ou NULL si l'information est absente)
        * Q3 = si un préfixe est indiqué dans l'étymologie, quel est ce préfixe ? (un seul mot ou NULL si l'information est absente)
        * Q4 = si un suffixe est indiqué dans l'étymologie, quel est ce suffixe ? (un seul mot ou NULL si l'information est absente)
        * Q5 = si des composants sont indiqués dans l'étymologie, quels sont ces composants ? (renvoie une liste de ces composants, sinon renvoie NULL)
        * Q6 = si le type du procédé morphologique est indiqué dans l'étymologie (suffixation, préfixation, composition, conversion, apocope, etc.), quel est ce type ? (un seul mot ou NULL si l'information est absente)
        Réponds uniquement avec ce de JSON :
        * 'Q1' : ' ',
        * 'Q2' : ' ',
        * 'Q3' : ' ',
        * 'Q4' : ' ',
        * 'Q5' : [ ],
        * 'Q6' : ' '
        Si l'information n'est pas explicitement présente dans l'étymologie ou la définition, répondre NULL. Ne jamais déduire ou supposer une information.
        """

    response = ollama.chat(
            model="gpt-oss",
            messages=[
                # system prompt
            {"role": "system",
              "content": SYSTEM_PROMPT},
                # contexte 1
            {"role": "user",
            "content": """l'enregistrement en JSON suivant : {'mot':'ouvrage', 'catégorie':'Nom commun', 'définition':['Travail ; action de travailler.'], 'étymologie':"['Dérivé de ouvrer, ancienne forme de œuvrer, avec le suffixe -age.']"} contient un mot du français, une catégorie, une définition et une étymologie. Construis un nouvel enregistrement JSON qui contient les réponses aux questions suivantes :
                    * Q1 = quelle est la langue d'origine du mot qui est indiquée dans l'étymologie ? (un seul mot ou null si l'information est absente)
                    * Q2 = quelle est la base du mot qui est indiquée dans l'étymologie ? (un seul mot ou null si l'information est absente)
                    * Q3 = si un préfixe est indiqué dans l'étymologie, quel est ce préfixe ? (un seul mot ou null si l'information est absente)
                    * Q4 = si un suffixe est indiqué dans l'étymologie, quel est ce suffixe ? (un seul mot ou null si l'information est absente)
                    * Q5 = si des composants sont indiqués dans l'étymologie, quels sont ces composants ? (renvoie une liste de ces composants, sinon renvoie null)
                    * Q6 = si le type du procédé morphologique est indiqué dans l'étymologie (suffixation, préfixation, composition, conversion, apocope, etc.), quel est ce type ? (un seul mot ou null si l'information est absente)
                    Réponds uniquement avec ce de JSON :
                    * 'Q1' : ' ',
                    * 'Q2' : ' ',
                    * 'Q3' : ' ',
                    * 'Q4' : ' ',
                    * 'Q5' : [ ],
                    * 'Q6' : ' '
                    Si l'information n'est pas explicitement présente dans l'étymologie ou la définition, répondre null. Ne jamais déduire ou supposer une information.
                    """},
                # réponse 1
            {"role": "assistant",
              "content": """{"Q1":"français", "Q2":"ouvrer", "Q3":null, "Q4":"-age", "Q5":null, "Q6":"suffixation"}"""},
                # contexte 2
            {"role": "user",
              "content": """l'enregistrement en JSON suivant : {'mot':'week', 'catégorie':'Nom commun', 'définition':['Week-end.'], 'étymologie':"['Apocope de week-end.']"} contient un mot du français, une catégorie, une définition et une étymologie. Construis un nouvel enregistrement JSON qui contient les réponses aux questions suivantes :
                    * Q1 = quelle est la langue d'origine du mot qui est indiquée dans l'étymologie ? (un seul mot ou null si l'information est absente)
                    * Q2 = quelle est la base du mot qui est indiquée dans l'étymologie ? (un seul mot ou null si l'information est absente)
                    * Q3 = si un préfixe est indiqué dans l'étymologie, quel est ce préfixe ? (un seul mot ou null si l'information est absente)
                    * Q4 = si un suffixe est indiqué dans l'étymologie, quel est ce suffixe ? (un seul mot ou null si l'information est absente)
                    * Q5 = si des composants sont indiqués dans l'étymologie, quels sont ces composants ? (renvoie une liste de ces composants, sinon renvoie null)
                    * Q6 = si le type du procédé morphologique est indiqué dans l'étymologie (suffixation, préfixation, composition, conversion, apocope, etc.), quel est ce type ? (un seul mot ou null si l'information est absente)
                    Réponds uniquement avec ce de JSON :
                    * 'Q1' : ' ',
                    * 'Q2' : ' ',
                    * 'Q3' : ' ',
                    * 'Q4' : ' ',
                    * 'Q5' : [ ],
                    * 'Q6' : ' '
                    Si l'information n'est pas explicitement présente dans l'étymologie ou la définition, répondre null. Ne jamais déduire ou supposer une information.
                    """},
                # réponse 2
              {"role": "assistant",
              "content": """{"Q1":null, "Q2":"week", "Q3":null, "Q4":null, "Q5":null, "Q6":"apocope"}"""},
                # prompt
              {"role": "user",
              "content": prompt},
            ],
            format="json"
        )
    print(f"Réponse : {response['message']['content']}")

    content = response["message"]["content"]
    data = json.loads(content)

    # besoin de upper les clés du dictionnaire data pour avoir "Q1" quand le modèle répond "q1"
    # permet d'éviter de rejeter trop de réponses qui ne correspondent pas à mon format
    new_data = {}

    for k, v in data.items() :
      new_data[k.upper()] = v

    data = new_data

    if list(data.keys())[0] == "Q1" and list(data.keys())[1] == "Q2" and list(data.keys())[2] == "Q3" and list(data.keys())[3] == "Q4" and list(data.keys())[4] == "Q5" and list(data.keys())[5] == "Q6" :
      llm_responses.append(data) # la réponse au bon format json est stockée dans la liste llm_responses
    else :
      bad_llm_responses.append(data)
      # si la réponse ne correspond pas au format demandé (trop de colonnes, pas assez) elle sera stockée dans la liste bad_llm_responses

  df_llm_responses = pd.DataFrame(llm_responses)
  # la liste des bonnes réponses du LLM est transformée en df pour pouvoir être plus facilement concaténé au small_df
  df_llm_responses_modified = df_llm_responses.rename(columns={"Q1": "lang", "Q2": "base", "Q3": "pref", "Q4": "suff", "Q5": "comps", "Q6": "type"})
  small_df_modified = small_df.rename(columns={"mot": "lemma", "catégorie": "cat", "définition": "def", "étymologie": "etym"}).reset_index(drop = True)
  small_llm_responses_concat = pd.concat([small_df_modified, df_llm_responses_modified], axis=1)

  if len(full_df) == 0 :
    full_df = small_llm_responses_concat
    full_df.to_csv("./tests_csv_few_shot.csv", index=False) # à adapter à Grid
  else :
    full_df = pd.concat( [full_df, small_llm_responses_concat], axis=0 ) # à adapter à Grid
    full_df.to_csv("./tests_csv_few_shot.csv", index=False)

{'mot': 'accueil', 'catégorie': 'Nom commun', 'définition': ['Cérémonie ou prestation réservée à un nouvel arrivant, consistant généralement à lui souhaiter la bienvenue et à l’aider dans son intégration ou ses démarches.'], 'étymologie': ['Déverbal de accueillir.']}
{'mot': 'lire', 'catégorie': 'Verbe', 'définition': ['Interpréter des informations écrites sous forme de mots ou de dessins sur un support.'], 'étymologie': ['Du latin lĕgĕre (« id. »), proprement «\u202frecueillir\u202f»']}
{'mot': 'lire', 'catégorie': 'Nom commun', 'définition': ['Monnaie utilisée en Italie avant l’usage de l’euro. Son symbole est ₤ (Unicode U+20A4).'], 'étymologie': ['De l’italien lira, du latin libra (« livre », le poids).']}
{'mot': 'encyclopédie', 'catégorie': 'Nom commun', 'définition': ['Ensemble, enchaînement de toutes les connaissances.'], 'étymologie': ['Du latin encyclopaedia forgé à la Renaissance, fausse lecture d’un manuscrit pour ἐγκύκλοπαιδεία ; mot tiré du grec ancien ἐγκύκλιος, énkúklios

# Champ messages à modifier

In [ ]:
messages=[
    # system prompt
 {"role": "system",
  "content": SYSTEM_PROMPT}
    # contexte 1
 {"role": "user",
 "content": """l'enregistrement en JSON suivant : {'mot':'ouvrage', 'catégorie':'Nom commun', 'définition':['Travail ; action de travailler.'], 'étymologie':"['Dérivé de ouvrer, ancienne forme de œuvrer, avec le suffixe -age.']"} contient un mot du français, une catégorie, une définition et une étymologie. Construis un nouvel enregistrement JSON qui contient les réponses aux questions suivantes :
        * Q1 = quelle est la langue d'origine du mot qui est indiquée dans l'étymologie ? (un seul mot ou null si l'information est absente)
        * Q2 = quelle est la base du mot qui est indiquée dans l'étymologie ? (un seul mot ou null si l'information est absente)
        * Q3 = si un préfixe est indiqué dans l'étymologie, quel est ce préfixe ? (un seul mot ou null si l'information est absente)
        * Q4 = si un suffixe est indiqué dans l'étymologie, quel est ce suffixe ? (un seul mot ou null si l'information est absente)
        * Q5 = si des composants sont indiqués dans l'étymologie, quels sont ces composants ? (renvoie une liste de ces composants, sinon renvoie null)
        * Q6 = si le type du procédé morphologique est indiqué dans l'étymologie (suffixation, préfixation, composition, conversion, apocope, etc.), quel est ce type ? (un seul mot ou null si l'information est absente)
        Réponds uniquement avec ce de JSON :
        * 'Q1' : ' ',
        * 'Q2' : ' ',
        * 'Q3' : ' ',
        * 'Q4' : ' ',
        * 'Q5' : [ ],
        * 'Q6' : ' '
        Si l'information n'est pas explicitement présente dans l'étymologie ou la définition, répondre null. Ne jamais déduire ou supposer une information.
        """},
    # réponse 1
 {"role": "assistant",
  "content": """{"Q1":"français", "Q2":"ouvrer", "Q3":null, "Q4":"-age", "Q5":null, "Q6":"suffixation"}"""},
    # contexte 2
 {"role": "user",
  "content": """l'enregistrement en JSON suivant : {'mot':'week', 'catégorie':'Nom commun', 'définition':['Week-end.'], 'étymologie':"['Apocope de week-end.']"} contient un mot du français, une catégorie, une définition et une étymologie. Construis un nouvel enregistrement JSON qui contient les réponses aux questions suivantes :
        * Q1 = quelle est la langue d'origine du mot qui est indiquée dans l'étymologie ? (un seul mot ou null si l'information est absente)
        * Q2 = quelle est la base du mot qui est indiquée dans l'étymologie ? (un seul mot ou null si l'information est absente)
        * Q3 = si un préfixe est indiqué dans l'étymologie, quel est ce préfixe ? (un seul mot ou null si l'information est absente)
        * Q4 = si un suffixe est indiqué dans l'étymologie, quel est ce suffixe ? (un seul mot ou null si l'information est absente)
        * Q5 = si des composants sont indiqués dans l'étymologie, quels sont ces composants ? (renvoie une liste de ces composants, sinon renvoie null)
        * Q6 = si le type du procédé morphologique est indiqué dans l'étymologie (suffixation, préfixation, composition, conversion, apocope, etc.), quel est ce type ? (un seul mot ou null si l'information est absente)
        Réponds uniquement avec ce de JSON :
        * 'Q1' : ' ',
        * 'Q2' : ' ',
        * 'Q3' : ' ',
        * 'Q4' : ' ',
        * 'Q5' : [ ],
        * 'Q6' : ' '
        Si l'information n'est pas explicitement présente dans l'étymologie ou la définition, répondre null. Ne jamais déduire ou supposer une information.
        """},
    # réponse 2
  {"role": "assistant",
  "content": """{"Q1":null, "Q2":"week", "Q3":null, "Q4":null, "Q5":null, "Q6":"apocope"}"""},
    # prompt
  {"role": "user",
   "content": prompt},
]